<a href="https://colab.research.google.com/github/SindhuMH/cartoon_generator/blob/image_generator_codes/scene_clip_qwen_gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import huggingface_hub

# This will attempt to log in using an existing token (e.g., from HF_TOKEN environment variable)
# or prompt you for a token if none is found.
# For a completely paste-free experience, ensure your HF_TOKEN environment variable is set
# before running this notebook.
huggingface_hub.login()

print("Hugging Face login initiated. Check the output for status or prompts.")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_ROOT = '/content/drive/MyDrive/panchatantra_assets'
import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q diffusers transformers accelerate safetensors ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 494.9 kB/s eta 0:00:00


In [3]:
import ftfy

# CLIP Vis Model Img Gen

In [5]:
import os, torch
from PIL import Image
from diffusers import AutoPipelineForText2Image, WanImageToVideoPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video
from transformers import CLIPVisionModel, CLIPVisionModelWithProjection

SDXL_MODEL='stabilityai/stable-diffusion-xl-base-1.0'
IP_REPO='h94/IP-Adapter'; IP_SUBFOLDER='sdxl_models'; IP_WEIGHT='ip-adapter-plus_sdxl_vit-h.safetensors'
WAN_MODEL='Wan-AI/Wan2.1-I2V-14B-480P-Diffusers'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SDXL_DTYPE=torch.float16 if DEVICE=='cuda' else torch.float32
WAN_DTYPE=torch.bfloat16 if DEVICE=='cuda' else torch.float32

def load_image(x):
    if isinstance(x, Image.Image): return x.convert('RGB')
    if not os.path.exists(x): raise FileNotFoundError(x)
    return Image.open(x).convert('RGB')

class SceneGenerator:
    def __init__(self): self.pipe=None

    # def load(self):
    #     self.pipe=AutoPipelineForText2Image.from_pretrained(SDXL_MODEL,torch_dtype=SDXL_DTYPE,use_safetensors=True).to(DEVICE)
    #     return self
    def load(self, num_references, action):
      # IMPORTANT:
      # SDXL IP-Adapter Plus requires the projection-based CLIP vision encoder.
      image_encoder = CLIPVisionModelWithProjection.from_pretrained(
          IP_REPO,
          subfolder="models/image_encoder",
          torch_dtype=SDXL_DTYPE,
      )
      self.pipe = AutoPipelineForText2Image.from_pretrained(
          SDXL_MODEL,
          image_encoder=image_encoder,
          torch_dtype=SDXL_DTYPE,
          use_safetensors=True,
          variant="fp16" if DEVICE == "cuda" else None,
      ).to(DEVICE)
      # Load one IP-Adapter for every reference image.
      self.pipe.load_ip_adapter(
          IP_REPO,
          subfolder="sdxl_models",
          weight_name=[IP_WEIGHT] * num_references,
      )
      self.action = action
      return self

    def generate(self, character_images, location_images, style_reference, action,
                 camera='wide cinematic shot', time_of_day='morning', output_path='scene.png',
                 seed=42, width=1344, height=768, steps=30):
        if self.pipe is None: self.load()
        chars=[load_image(x) for x in character_images]; locs=[load_image(x) for x in location_images]#; style=load_image(style_reference)
        if not chars: raise ValueError('Provide at least one character image.')
        if not locs: raise ValueError('Provide at least one location image.')
        refs=[[x] for x in chars]+[[x] for x in locs]#+[[style]]
        self.pipe.load_ip_adapter(IP_REPO,subfolder=IP_SUBFOLDER,weight_name=[IP_WEIGHT]*len(refs))
        self.pipe.set_ip_adapter_scale([0.75]*len(chars)+[0.45]*len(locs))
        prompt=f'''Ultra High quality 2D children's storybook cartoon scene.
        scene: {self.action}
        Characters: {len(chars)} referenced characters.
        Environments: {len(locs)} referenced environments.
        Action: {action}
        Camera: {camera}
        Time: {time_of_day}
        Create ONE coherent scene containing all requested characters. Preserve each character's identity, species, face, body proportions, colors, clothing and distinctive features from its reference image. Combine the referenced environments naturally into one coherent setting. All characters must be clearly visible and naturally positioned. Clean polished children's animation artwork, expressive faces, soft lighting, cinematic composition, consistent visual style.'''
        negative='''photorealistic, extra characters, duplicate characters, merged characters, wrong species, extra limbs, bad anatomy, cropped characters, separate panels, collage, split screen, text, watermark, logo, blurry, low quality'''
        result=self.pipe(prompt=prompt,negative_prompt=negative,ip_adapter_image=refs,width=width,height=height,num_inference_steps=steps,guidance_scale=6.0,generator=torch.Generator(device=DEVICE).manual_seed(seed)).images[0]
        result.save(output_path); return result

class WanGenerator:
    def __init__(self): self.pipe=None
    def load(self):
        encoder=CLIPVisionModel.from_pretrained(WAN_MODEL,subfolder='image_encoder',torch_dtype=torch.float32)
        vae=AutoencoderKLWan.from_pretrained(WAN_MODEL,subfolder='vae',torch_dtype=torch.float32)
        self.pipe=WanImageToVideoPipeline.from_pretrained(WAN_MODEL,vae=vae,image_encoder=encoder,torch_dtype=WAN_DTYPE).to(DEVICE)
        return self
    def generate(self,scene_image,motion_prompt,output_path='scene.mp4',width=832,height=480,num_frames=49,fps=16,steps=30):
        if self.pipe is None: self.load()
        output=self.pipe(image=load_image(scene_image),prompt=motion_prompt,width=width,height=height,num_frames=num_frames,guidance_scale=5.0,num_inference_steps=steps)
        frames=output.frames[0]; export_to_video(frames,output_path,fps=fps); return frames

def generate_scene_and_clip(character_images,location_images,style_reference,action,motion_prompt,output_scene='scene.png',output_video='scene.mp4',camera='wide cinematic shot',time_of_day='morning',seed=42):
    num_references = (
      len(character_images)
      + len(location_images)
      # + (1 if style_reference is not None else 0)
      )
    print("creating just the scene")
    sg=SceneGenerator().load(num_references, action); sg.generate(character_images,location_images,style_reference,action,camera,time_of_day,output_scene,seed)
    print(f"scene_image created {sg}")
    del sg.pipe
    # if torch.cuda.is_available(): torch.cuda.empty_cache()
    # WanGenerator().load().generate(output_scene,motion_prompt,output_video)
    # print(f"video created")
    return True

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


In [19]:
parent_dir = OUTPUT_ROOT+'/story1'
status = generate_scene_and_clip(
      character_images=[parent_dir+'/characters/chintu.png',parent_dir+'/characters/kalu.png',parent_dir+'/characters/mina.png'],
      location_images=[parent_dir+'/river_bank.jpeg',parent_dir+'/forest_path.jpeg'],
      style_reference=parent_dir+'/style_reference.png',
      action='Chintu and Kalu walk beside the river while Mina stands near the forest path and waves to them. All three interact happily.',
      motion_prompt='Chintu and Kalu slowly walk and talk. Mina gently waves. Their expressions move naturally. Grass, leaves and river water move gently in the breeze. The camera slowly pushes forward. Smooth children animation. Keep all character appearances consistent with the input scene.',
      output_scene=parent_dir+'/scene_img/scene_003.png',output_video=parent_dir+'/clips/scene_001.mp4',camera='wide shot showing all three characters',time_of_day='soft morning light',seed=123)


creating just the scene


Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (171 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ["waves to them . all three interact happily . camera : wide shot showing all three characters time : soft morning light create one coherent scene containing all requested characters . preserve each character 's identity , species , face , body proporti

  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


scene_image created <__main__.SceneGenerator object at 0x7de1a046a120>


In [ ]:
import os
import torch
from PIL import Image
from diffusers import QwenImageEditPlusPipeline


class SceneGenerator:

    MODEL_ID = "Qwen/Qwen-Image-Edit-2509"

    def __init__(self, device="cuda"):
        self.device = device
        self.pipeline = None

    def load(self):
        print("Loading Qwen Image Edit Plus...")

        self.pipeline = QwenImageEditPlusPipeline.from_pretrained(
            self.MODEL_ID,
            torch_dtype=torch.bfloat16,
        )

        self.pipeline.to(self.device)

        self.pipeline.set_progress_bar_config(disable=None)

        print("Qwen pipeline loaded.")

        return self

    def load_images(self, image_paths):
        """
        Load character/location/style reference images.

        image_paths:
            list of image file paths
        """

        images = []

        for path in image_paths:
            if not os.path.exists(path):
                raise FileNotFoundError(f"Image not found: {path}")

            images.append(Image.open(path).convert("RGB"))

        return images

    def generate_scene(
        self,
        image_paths,
        prompt,
        output_path,
        seed=0,
        num_inference_steps=40,
        true_cfg_scale=4.0,
        negative_prompt=" ",
    ):

        if self.pipeline is None:
            self.load()

        # Load all references
        images = self.load_images(image_paths)

        print(f"Using {len(images)} reference images")

        inputs = {
            "image": images,
            "prompt": prompt,
            "generator": torch.manual_seed(seed),
            "true_cfg_scale": true_cfg_scale,
            "negative_prompt": negative_prompt,
            "num_inference_steps": num_inference_steps,
            "guidance_scale": 1.0,
            "num_images_per_prompt": 1,
        }

        print("Generating scene...")

        with torch.inference_mode():
            output = self.pipeline(**inputs)

        output_image = output.images[0]

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        output_image.save(output_path)

        print(f"Scene saved to: {output_path}")

        return output_image

#Qwen Img Gen

In [7]:
import os
import torch
from PIL import Image

from diffusers import (
    QwenImageEditPlusPipeline,
    WanImageToVideoPipeline,
    AutoencoderKLWan,
)
from diffusers.utils import export_to_video
from transformers import CLIPVisionModel


QWEN_MODEL = "Qwen/Qwen-Image-Edit-2509"
WAN_MODEL = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QWEN_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
WAN_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32


def load_image(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if not os.path.exists(x):
        raise FileNotFoundError(x)

    return Image.open(x).convert("RGB")


class SceneGenerator:

    def __init__(self):
        self.pipe = None

    def load(self):

        print("Loading Qwen Image Edit Plus...")

        self.pipe = QwenImageEditPlusPipeline.from_pretrained(
            QWEN_MODEL,
            torch_dtype=QWEN_DTYPE,
        )

        # self.pipe.to(DEVICE)
        # IMPORTANT: don't put the entire model on GPU
        self.pipe.enable_model_cpu_offload()

        # Saves additional VRAM during VAE operations
        # self.pipe.enable_vae_slicing()

        self.pipe.set_progress_bar_config(disable=None)

        print("Qwen pipeline loaded.")

        return self

    def generate(
        self,
        character_images,
        location_images,
        style_reference,
        action,
        camera="wide cinematic shot",
        time_of_day="morning",
        output_path="scene.png",
        seed=42,
        width=1344,
        height=768,
        steps=40,
    ):

        if self.pipe is None:
            self.load()

        # --------------------------------------------------
        # Load reference images
        # --------------------------------------------------

        chars = [load_image(x) for x in character_images]
        locs = [load_image(x) for x in location_images]

        if not chars:
            raise ValueError("Provide at least one character image.")

        if not locs:
            raise ValueError("Provide at least one location image.")

        # --------------------------------------------------
        # Combine all references
        #
        # Qwen accepts multiple images directly.
        # --------------------------------------------------

        reference_images = chars + locs

        # Optionally include style reference
        if style_reference is not None:
            reference_images.append(load_image(style_reference))

        # --------------------------------------------------
        # Prompt
        # --------------------------------------------------

        prompt = f"""
Create ONE coherent 2D children's storybook cartoon scene.

ACTION:
{action}

CHARACTERS:
There are {len(chars)} referenced characters.

ENVIRONMENTS:
There are {len(locs)} referenced environment images.

CAMERA:
{camera}

TIME OF DAY:
{time_of_day}

IMPORTANT CHARACTER INSTRUCTIONS:

Preserve the identity and visual appearance of every
character from the provided reference images.

Preserve:
- face
- eyes
- species
- body proportions
- colors
- clothing
- hairstyle/fur
- distinctive physical features

All referenced characters must appear together
in the SAME scene.

Do not duplicate characters.

IMPORTANT ENVIRONMENT INSTRUCTIONS:

Use the provided environment references to create
ONE coherent environment.

Do not create separate panels.
Do not create a collage.
Do not split the image into sections.

The characters should naturally exist inside
the environment.

SCENE COMPOSITION:

{action}

All characters should be clearly visible,
naturally positioned and interacting with the scene.

Style:
clean polished children's animation artwork,
storybook illustration, expressive faces,
soft lighting, cinematic composition,
consistent visual style.
"""

        negative_prompt = """
photorealistic,
extra characters,
duplicate characters,
merged characters,
wrong species,
different character identity,
extra limbs,
missing limbs,
bad anatomy,
deformed face,
cropped characters,
separate panels,
collage,
split screen,
multiple scenes,
text,
watermark,
logo,
blurry,
low quality
"""

        # --------------------------------------------------
        # Qwen generation
        # --------------------------------------------------

        inputs = {
            "image": reference_images,
            "prompt": prompt,
            "generator": torch.manual_seed(seed),
            "true_cfg_scale": 4.0,
            "negative_prompt": negative_prompt,
            "num_inference_steps": steps,
            "guidance_scale": 1.0,
            "num_images_per_prompt": 1,
        }

        print(
            f"Generating scene using "
            f"{len(reference_images)} reference images..."
        )

        with torch.inference_mode():

            output = self.pipe(**inputs)

        result = output.images[0]

        # Resize if required
        if width is not None and height is not None:
            result = result.resize((width, height))

        os.makedirs(
            os.path.dirname(output_path)
            if os.path.dirname(output_path)
            else ".",
            exist_ok=True,
        )

        result.save(output_path)

        print(f"Scene image created: {output_path}")

        return result


class WanGenerator:

    def __init__(self):
        self.pipe = None

    def load(self):

        encoder = CLIPVisionModel.from_pretrained(
            WAN_MODEL,
            subfolder="image_encoder",
            torch_dtype=torch.float32,
        )

        vae = AutoencoderKLWan.from_pretrained(
            WAN_MODEL,
            subfolder="vae",
            torch_dtype=torch.float32,
        )

        self.pipe = WanImageToVideoPipeline.from_pretrained(
            WAN_MODEL,
            vae=vae,
            image_encoder=encoder,
            torch_dtype=WAN_DTYPE,
        ).to(DEVICE)

        return self

    def generate(
        self,
        scene_image,
        motion_prompt,
        output_path="scene.mp4",
        width=832,
        height=480,
        num_frames=49,
        fps=16,
        steps=30,
    ):

        if self.pipe is None:
            self.load()

        output = self.pipe(
            image=load_image(scene_image),
            prompt=motion_prompt,
            width=width,
            height=height,
            num_frames=num_frames,
            guidance_scale=5.0,
            num_inference_steps=steps,
        )

        frames = output.frames[0]

        export_to_video(
            frames,
            output_path,
            fps=fps,
        )

        return frames


def generate_scene_and_clip(
    character_images,
    location_images,
    style_reference,
    action,
    motion_prompt,
    output_scene="scene.png",
    output_video="scene.mp4",
    camera="wide cinematic shot",
    time_of_day="morning",
    seed=42,
):

    print("Creating scene with Qwen...")

    sg = SceneGenerator().load()

    sg.generate(
        character_images=character_images,
        location_images=location_images,
        style_reference=style_reference,
        action=action,
        camera=camera,
        time_of_day=time_of_day,
        output_path=output_scene,
        seed=seed,
    )

    print(f"Scene image created: {output_scene}")

    # Free Qwen before loading Wan
    del sg.pipe
    del sg

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------
    # Wan video generation
    # --------------------------------------------------

    print("Creating video with Wan...")

    wg = WanGenerator().load()

    wg.generate(
        scene_image=output_scene,
        motion_prompt=motion_prompt,
        output_path=output_video,
    )

    print(f"Video created: {output_video}")

    del wg.pipe
    del wg

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return True

In [8]:
parent_dir = OUTPUT_ROOT+'/story1'
# status = generate_scene_and_clip(
#       character_images=[parent_dir+'/characters/chintu.png',parent_dir+'/characters/kalu.png',parent_dir+'/characters/mina.png'],
#       location_images=[parent_dir+'/river_bank.jpeg',parent_dir+'/forest_path.jpeg'],
#       style_reference=parent_dir+'/style_reference.png',
#       action='Chintu and Kalu walk beside the river while Mina stands near the forest path and waves to them. All three interact happily.',
#       motion_prompt='Chintu and Kalu slowly walk and talk. Mina gently waves. Their expressions move naturally. Grass, leaves and river water move gently in the breeze. The camera slowly pushes forward. Smooth children animation. Keep all character appearances consistent with the input scene.',
#       output_scene=parent_dir+'/scene_img/scene_003.png',output_video=parent_dir+'/clips/scene_001.mp4',camera='wide shot showing all three characters',time_of_day='soft morning light',seed=123)

generate_scene_and_clip(
    character_images=[
        parent_dir + "/chintu.jpeg",
        parent_dir + "/kalu.jpeg",
        parent_dir + "/mina.jpeg",
    ],

    location_images=[
        parent_dir + "/river_bank.jpeg",
        parent_dir + "/forest_path.jpeg",
    ],

    style_reference=parent_dir + "/style_reference.png",

    action="""
    Chintu is standing on the river bank.
    Kalu is sitting near the water.
    Mina is walking toward them.
    They are looking at each other and having a friendly conversation.
    """,

    motion_prompt="""
    The characters gently move while talking.
    Their facial expressions subtly change.
    The river water moves gently.
    Leaves and grass move slightly in the breeze.
    Camera slowly pushes forward.
    """,

    output_scene=parent_dir + "/scene_001.png",

    output_video=parent_dir + "/scene_001.mp4",

    camera="wide cinematic shot",

    time_of_day="morning",

    seed=42,
)

Creating scene with Qwen...
Loading Qwen Image Edit Plus...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Qwen pipeline loaded.
Generating scene using 6 reference images...


...

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Video created: /content/drive/MyDrive/panchatantra_assets/story1/scene_001.mp4


True